# RecKAN vs. Baselines — ETTh1 Forecasting (Seeds 1,2,42)

In [ ]:
# =====================================================================
# RECKAN VS 3 BASELINE KANS - ETTh1 FORECASTING
# Testing on seed=1, 2, 42
# =====================================================================
!pip install aeon -q
import os
import csv
import time
import math
import argparse

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# =====================================================================
# PART 1 — MODELS (Improved RecKAN)
# =====================================================================

class RecursiveBasis(nn.Module):
    def __init__(self, K, B=3.0, init=(0.0, 0.5, 0.0, 0.0, -0.5)):
        super().__init__()
        self.K = K
        self.B = B
        atanh = lambda v: 0.5 * math.log((1 + v / B) / (1 - v / B)) if abs(v) < B else 0.0
        self.a_raw = nn.Parameter(torch.tensor(atanh(init[0])))
        self.b_raw = nn.Parameter(torch.tensor(atanh(init[1])))
        self.c_raw = nn.Parameter(torch.tensor(atanh(init[2])))
        self.d_raw = nn.Parameter(torch.tensor(atanh(init[3])))
        self.e_raw = nn.Parameter(torch.tensor(atanh(init[4])))

    def forward(self, x):
        a = self.B * torch.tanh(self.a_raw)
        b = self.B * torch.tanh(self.b_raw)
        c = self.B * torch.tanh(self.c_raw)
        d = self.B * torch.tanh(self.d_raw)
        e = self.B * torch.tanh(self.e_raw)
        
        R0 = torch.zeros_like(x)
        R1 = torch.ones_like(x)
        basis = [R0, R1]
        
        for n in range(1, self.K):
            alpha = a * x ** 2 + b * x + c
            beta = d * x + e
            Rn1 = alpha * basis[-1] + beta * basis[-2]
            basis.append(Rn1)
            
        return torch.stack(basis, dim=-1)


class RecKANLayer(nn.Module):
    def __init__(self, d_in, d_out, K, basis: RecursiveBasis):
        super().__init__()
        self.d_in, self.d_out, self.K = d_in, d_out, K
        self.basis = basis
        self.W = nn.Parameter(torch.randn(d_in, d_out, K + 1) * (1.0 / math.sqrt(d_in * (K + 1))))

    def forward(self, x):
        x_clamped = x.clamp(-2.0, 2.0)
        R = self.basis(x_clamped)
        y = torch.einsum('bik,iok->bo', R, self.W)
        return y


class RecKAN(nn.Module):
    def __init__(self, dims, K, use_norm=True, dropout_rate=0.1):
        super().__init__()
        self.basis = RecursiveBasis(K)
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.use_norm = use_norm
        
        for i in range(len(dims) - 1):
            self.layers.append(RecKANLayer(dims[i], dims[i + 1], K, self.basis))
            if i < len(dims) - 2:
                if use_norm:
                    self.norms.append(nn.LayerNorm(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout_rate))
            else:
                if use_norm:
                    self.norms.append(nn.Identity())
                self.drops.append(nn.Identity())

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.norms[i](x)
                x = self.drops[i](x)
                x = torch.tanh(x)
        return x


class ChebyKANLayer(nn.Module):
    def __init__(self, d_in, d_out, K):
        super().__init__()
        self.d_in, self.d_out, self.K = d_in, d_out, K
        self.W = nn.Parameter(torch.randn(d_in, d_out, K + 1) * (1.0 / math.sqrt(d_in * (K + 1))))

    def forward(self, x):
        h = x.clamp(-2.0, 2.0)
        T0 = torch.ones_like(h)
        T1 = h
        basis = [T0, T1]
        for n in range(1, self.K):
            Tn1 = 2 * h * basis[-1] - basis[-2]
            basis.append(Tn1)
        R = torch.stack(basis, dim=-1)
        y = torch.einsum('bik,iok->bo', R, self.W)
        return y


class ChebyKAN(nn.Module):
    def __init__(self, dims, K, use_norm=True, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.use_norm = use_norm
        
        for i in range(len(dims) - 1):
            self.layers.append(ChebyKANLayer(dims[i], dims[i + 1], K))
            if i < len(dims) - 2:
                if use_norm:
                    self.norms.append(nn.LayerNorm(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout_rate))
            else:
                if use_norm:
                    self.norms.append(nn.Identity())
                self.drops.append(nn.Identity())

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.norms[i](x)
                x = self.drops[i](x)
                x = torch.tanh(x)
        return x


class JacobiShape(nn.Module):
    def __init__(self, init_alpha=0.0, init_beta=0.0, B=2.0):
        super().__init__()
        self.B = B
        atanh = lambda v: 0.5 * math.log((1 + v / B) / (1 - v / B)) if abs(v) < B else 0.0
        self.alpha_raw = nn.Parameter(torch.tensor(atanh(init_alpha)))
        self.beta_raw = nn.Parameter(torch.tensor(atanh(init_beta)))

    def get(self):
        return self.B * torch.tanh(self.alpha_raw), self.B * torch.tanh(self.beta_raw)


class JacobiKANLayer(nn.Module):
    def __init__(self, d_in, d_out, K, shape: JacobiShape):
        super().__init__()
        self.d_in, self.d_out, self.K = d_in, d_out, K
        self.shape = shape
        self.W = nn.Parameter(torch.randn(d_in, d_out, K + 1) * (1.0 / math.sqrt(d_in * (K + 1))))

    def forward(self, x):
        h = x.clamp(-2.0, 2.0)
        al, be = self.shape.get()
        P0 = torch.ones_like(h)
        basis = [P0]
        if self.K >= 1:
            P1 = 0.5 * (al - be) + 0.5 * (al + be + 2) * h
            basis.append(P1)
        for n in range(1, self.K):
            n_ = float(n)
            a1 = 2 * (n_ + 1) * (n_ + al + be + 1) * (2 * n_ + al + be)
            a2 = (2 * n_ + al + be + 1) * (al ** 2 - be ** 2)
            a3 = (2 * n_ + al + be) * (2 * n_ + al + be + 1) * (2 * n_ + al + be + 2)
            a4 = 2 * (n_ + al) * (n_ + be) * (2 * n_ + al + be + 2)
            a1 = a1 + 1e-4
            Pn1 = ((a2 + a3 * h) * basis[-1] - a4 * basis[-2]) / a1
            basis.append(Pn1)
        R = torch.stack(basis, dim=-1)
        y = torch.einsum('bik,iok->bo', R, self.W)
        return y


class JacobiKAN(nn.Module):
    def __init__(self, dims, K, use_norm=True, dropout_rate=0.1):
        super().__init__()
        self.shape = JacobiShape()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.use_norm = use_norm
        
        for i in range(len(dims) - 1):
            self.layers.append(JacobiKANLayer(dims[i], dims[i + 1], K, self.shape))
            if i < len(dims) - 2:
                if use_norm:
                    self.norms.append(nn.LayerNorm(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout_rate))
            else:
                if use_norm:
                    self.norms.append(nn.Identity())
                self.drops.append(nn.Identity())

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.norms[i](x)
                x = self.drops[i](x)
                x = torch.tanh(x)
        return x


def bspline_basis(x, grid, spline_order):
    x = x.unsqueeze(-1)
    bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).float()
    for k in range(1, spline_order + 1):
        left_num = x - grid[:, :-(k + 1)]
        left_den = grid[:, k:-1] - grid[:, :-(k + 1)]
        left = left_num / left_den.clamp_min(1e-6) * bases[:, :, :-1]
        right_num = grid[:, k + 1:] - x
        right_den = grid[:, k + 1:] - grid[:, 1:-k]
        right = right_num / right_den.clamp_min(1e-6) * bases[:, :, 1:]
        bases = left + right
    return bases


class SplineKANLayer(nn.Module):
    def __init__(self, d_in, d_out, grid_size, spline_order=3, grid_range=(-1.2, 1.2)):
        super().__init__()
        self.d_in, self.d_out = d_in, d_out
        self.grid_size, self.spline_order = grid_size, spline_order
        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = torch.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0]
        grid = grid.unsqueeze(0).repeat(d_in, 1)
        self.register_buffer('grid', grid)
        n_basis = grid_size + spline_order
        self.spline_W = nn.Parameter(torch.randn(d_in, d_out, n_basis) * (1.0 / math.sqrt(d_in * n_basis)))
        self.base_W = nn.Parameter(torch.randn(d_in, d_out) * (1.0 / math.sqrt(d_in)))
        self.scaler = nn.Parameter(torch.ones(d_in, d_out))

    def forward(self, x):
        h = x.clamp(-2.0, 2.0)
        base = torch.nn.functional.silu(h)
        base_out = base @ self.base_W
        B_ = bspline_basis(h, self.grid, self.spline_order)
        spline_out = torch.einsum('bik,iok->bio', B_, self.spline_W)
        spline_out = (spline_out * self.scaler).sum(dim=1)
        return base_out + spline_out


class SplineKAN(nn.Module):
    def __init__(self, dims, grid_size, spline_order=3, use_norm=True, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.use_norm = use_norm
        
        for i in range(len(dims) - 1):
            self.layers.append(SplineKANLayer(dims[i], dims[i + 1], grid_size, spline_order))
            if i < len(dims) - 2:
                if use_norm:
                    self.norms.append(nn.LayerNorm(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout_rate))
            else:
                if use_norm:
                    self.norms.append(nn.Identity())
                self.drops.append(nn.Identity())

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.norms[i](x)
                x = self.drops[i](x)
                x = torch.tanh(x)
        return x


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def match_spline_grid(dims, target_params, spline_orders=(1, 2, 3), search_range=range(1, 60)):
    best = None
    for order in spline_orders:
        for g in search_range:
            if g < 1:
                continue
            m = SplineKAN(dims, g, order, use_norm=False)
            p = count_params(m)
            diff = abs(p - target_params)
            if best is None or diff < best[0]:
                best = (diff, order, g)
    return best[1], best[2]


# =====================================================================
# PART 2 — DATASET (ETTh1)
# =====================================================================

DATA_DIR = os.environ.get('RECKAN_DATA_DIR', './data')
os.makedirs(DATA_DIR, exist_ok=True)


def etth1(seed=0, lookback=96, horizon=1, test_frac=0.2):
    """Real ETTh1 (Electricity Transformer Temperature) forecasting"""
    import pandas as pd
    url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
    df = pd.read_csv(url)
    feats = df.drop(columns=['date']).values.astype(np.float32)
    target_col = feats.shape[1] - 1

    sx = StandardScaler().fit(feats)
    feats_n = sx.transform(feats).astype(np.float32)

    X, y = [], []
    for t in range(len(feats_n) - lookback - horizon + 1):
        X.append(feats_n[t:t + lookback].reshape(-1))
        y.append(feats_n[t + lookback:t + lookback + horizon, target_col])
    X = np.stack(X).astype(np.float32)
    y = np.stack(y).astype(np.float32)

    n_test = int(len(X) * test_frac)
    Xtr, Xte = X[:-n_test], X[-n_test:]
    ytr, yte = y[:-n_test], y[-n_test:]
    return (torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xte), torch.tensor(yte),
            'regression', y.shape[1])


# =====================================================================
# PART 3 — TRAINING / COMPARISON
# =====================================================================

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def iterate_batches(X, y, batch_size, shuffle=True):
    n = X.shape[0]
    idx = torch.randperm(n) if shuffle else torch.arange(n)
    for i in range(0, n, batch_size):
        b = idx[i:i + batch_size]
        yield X[b], y[b]


def train_one(model, Xtr, ytr, Xte, yte, task, epochs, batch_size, lr, weight_decay=1e-4, model_name="", seed=0):
    model.to(DEVICE)
    Xtr, ytr, Xte, yte = Xtr.to(DEVICE), ytr.to(DEVICE), Xte.to(DEVICE), yte.to(DEVICE)
    
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss() if task == 'classification' else nn.MSELoss()

    train_losses, test_metrics = [], []
    t0 = time.time()
    
    for ep in range(epochs):
        model.train()
        ep_loss, n_batches = 0.0, 0
        for xb, yb in iterate_batches(Xtr, ytr, batch_size):
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            ep_loss += loss.item()
            n_batches += 1
        train_losses.append(ep_loss / max(n_batches, 1))

        model.eval()
        with torch.no_grad():
            preds = []
            for xb, _ in iterate_batches(Xte, yte, batch_size, shuffle=False):
                preds.append(model(xb))
            preds = torch.cat(preds, dim=0)
            if task == 'classification':
                metric = (preds.argmax(dim=1) == yte).float().mean().item()
            else:
                metric = nn.functional.mse_loss(preds, yte).item()
        test_metrics.append(metric)
        
        scheduler.step()
        
        if (ep + 1) % 10 == 0:
            print(f'    [{model_name}] Epoch {ep+1}/{epochs} | Train Loss: {train_losses[-1]:.6f} | Test MSE: {metric:.6f}')

    train_time = time.time() - t0
    final_metric = test_metrics[-1]
    best_metric = min(test_metrics)
    
    print(f'    [{model_name}] FINAL | Test MSE: {final_metric:.6f} | Best MSE: {best_metric:.6f} | Time: {train_time:.1f}s')
    
    return train_losses, test_metrics, train_time, final_metric, best_metric


def build_model(name, dims, K, target_params, use_norm=True, dropout_rate=0.1):
    if name == 'SplineKAN':
        order, grid = match_spline_grid(dims, target_params)
        return SplineKAN(dims, grid, order, use_norm=use_norm, dropout_rate=dropout_rate)
    elif name == 'RecKAN':
        return RecKAN(dims, K, use_norm=use_norm, dropout_rate=dropout_rate)
    elif name == 'ChebyKAN':
        return ChebyKAN(dims, K, use_norm=use_norm, dropout_rate=dropout_rate)
    elif name == 'JacobiKAN':
        return JacobiKAN(dims, K, use_norm=use_norm, dropout_rate=dropout_rate)
    else:
        raise ValueError(f"Unknown model: {name}")


def run_experiment(seed, model_names, all_results):
    print(f'\n{"="*60}')
    print(f'SEED = {seed}')
    print(f'{"="*60}')
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    Xtr, ytr, Xte, yte, task, out_dim = etth1(seed=seed)
    in_dim = Xtr.shape[1]
    
    hidden1 = 32
    hidden2 = 16
    dims = [in_dim, hidden1, hidden2, out_dim]
    K = 3
    epochs = 60
    batch_size = 64
    lr = 1e-3
    weight_decay = 1e-4
    dropout = 0.1
    use_norm = True
    
    print(f'  input_dim={in_dim}  output_dim={out_dim}  task={task}  '
          f'n_train={len(Xtr)}  n_test={len(Xte)}')

    ref = RecKAN(dims, K, use_norm=use_norm, dropout_rate=dropout)
    target_params = count_params(ref)
    print(f'  target parameter budget (RecKAN reference): {target_params}')
    print('-'*60)

    for mname in model_names:
        model = build_model(mname, dims, K, target_params, 
                           use_norm=use_norm, dropout_rate=dropout)
        n_params = count_params(model)
        print(f'\n  Training {mname:12s} | params={n_params}')
        
        train_losses, test_metrics, train_time, final_metric, best_metric = train_one(
            model, Xtr, ytr, Xte, yte, task,
            epochs=epochs, batch_size=batch_size, 
            lr=lr, weight_decay=weight_decay,
            model_name=mname, seed=seed)
        
        all_results.append(dict(
            seed=seed,
            model=mname,
            task=task,
            n_params=n_params,
            final_test_mse=final_metric,
            best_test_mse=best_metric,
            train_time_sec=train_time))
        
        print(f'  DONE {mname:12s} | Best MSE: {best_metric:.6f}')
        print('-'*60)


def main():
    model_names = ['RecKAN', 'ChebyKAN', 'JacobiKAN', 'SplineKAN']
    seeds = [1, 2, 42]
    
    all_results = []
    for seed in seeds:
        run_experiment(seed, model_names, all_results)
    
    print('\n' + '='*80)
    print('FINAL SUMMARY TABLE')
    print('='*80)
    print(f"{'Seed':>6s} {'Model':12s} {'Params':>10s} {'Final MSE':>12s} {'Best MSE':>12s} {'Time(s)':>10s}")
    print('-'*80)
    for r in all_results:
        print(f"{r['seed']:6d} {r['model']:12s} {r['n_params']:10d} "
              f"{r['final_test_mse']:12.6f} {r['best_test_mse']:12.6f} "
              f"{r['train_time_sec']:10.1f}")
    print('='*80)
    
    print('\n' + '='*80)
    print('MEAN BEST MSE ACROSS SEEDS')
    print('='*80)
    print(f"{'Model':12s} {'Mean Best MSE':>16s} {'Std':>12s}")
    print('-'*80)
    for model in model_names:
        best_mses = [r['best_test_mse'] for r in all_results if r['model'] == model]
        mean_mse = np.mean(best_mses)
        std_mse = np.std(best_mses)
        print(f"{model:12s} {mean_mse:16.6f} {std_mse:12.6f}")
    print('='*80)
    
    print('\n' + '='*80)
    print('BEST MODEL BY MEAN BEST MSE')
    print('='*80)
    best_mses_by_model = {}
    for model in model_names:
        best_mses = [r['best_test_mse'] for r in all_results if r['model'] == model]
        best_mses_by_model[model] = np.mean(best_mses)
    
    best_model = min(best_mses_by_model, key=best_mses_by_model.get)
    print(f'Best model: {best_model} with Mean Best MSE = {best_mses_by_model[best_model]:.6f}')
    print('='*80)


if __name__ == '__main__':
    main()